# Feature Engineering

**Objective:** Transform raw attributes and domain knowledge into numerical, high-signal features ready for machine learning models (Phase 4).

### Key Transformations in this Phase:
1. **Applicant Age:** Compute age at disbursal (in years) from `Date.of.Birth`.
2. **Credit History Tenure:** Convert duration strings (`'1yrs 11mon'`) into total numeric months.
3. **Domain Financial Features:**
   - `is_new_to_credit`: Binary flag identifying first-time borrowers without bureau history.
   - `down_payment`: Customer equity invested (`asset_cost - disbursed_amount`).
4. **Bureau Risk Tiers:** Consolidate 20 verbose CRISIL descriptions into 6 standardized risk tiers.
5. **One-Hot Encoding:** Convert categorical columns into clean numeric binary features.
6. **Export:** Save final ML-ready feature matrix to `data/train_features.csv`.

### Step 1: Import Libraries and Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

# Load cleaned dataset from Phase 2
CLEAN_PATH = Path("../data/train_cleaned.csv") if Path("../data/train_cleaned.csv").exists() else Path("data/train_cleaned.csv")
df = pd.read_csv(CLEAN_PATH)

print(f"Cleaned dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(2)

Cleaned dataset loaded: 233,154 rows x 37 columns


,disbursed_amount,asset_cost,ltv,branch_id,manufacturer_id,Date.of.Birth,Employment.Type,DisbursalDate,State_ID,MobileNo_Avl_Flag,Aadhar_flag,PAN_flag,VoterID_flag,Driving_flag,Passport_flag,PERFORM_CNS.SCORE,PERFORM_CNS.SCORE.DESCRIPTION,PRI.NO.OF.ACCTS,PRI.ACTIVE.ACCTS,PRI.OVERDUE.ACCTS,PRI.CURRENT.BALANCE,PRI.SANCTIONED.AMOUNT,PRI.DISBURSED.AMOUNT,SEC.NO.OF.ACCTS,SEC.ACTIVE.ACCTS,SEC.OVERDUE.ACCTS,SEC.CURRENT.BALANCE,SEC.SANCTIONED.AMOUNT,SEC.DISBURSED.AMOUNT,PRIMARY.INSTAL.AMT,SEC.INSTAL.AMT,NEW.ACCTS.IN.LAST.SIX.MONTHS,DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS,AVERAGE.ACCT.AGE,CREDIT.HISTORY.LENGTH,NO.OF_INQUIRIES,loan_default
0,50578,58400,89.55,67,45,01-01-84,Salaried,03-08-18,6,1,1,0,0,0,0,0,No Bureau History Available,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0
1,47145,65550,73.23,67,45,31-07-85,Self employed,26-09-18,6,1,1,0,0,0,0,598,I-Medium Risk,1,1,1,27600,50200,50200,0,0,0,0,0,0,1991,0,0,1,1yrs 11mon,1yrs 11mon,0,1


### Step 2: Applicant Age Calculation

 Dates cannot be fed directly into ML models. We extract the applicant's age at the time of loan disbursal (2018). We handle 2-digit years by assigning years > 20 to the 1900s and <= 20 to the 2000s.

In [2]:
def calculate_age(dob_str):
    d, m, y = map(int, dob_str.split('-'))
    birth_year = 1900 + y if y > 20 else 2000 + y
    return 2018 - birth_year  # Loans disbursed in 2018

df['age'] = df['Date.of.Birth'].apply(calculate_age)

print("Applicant Age Distribution Summary:")
print(df['age'].describe())
assert df['age'].min() >= 18, "Found underage applicants!"

Applicant Age Distribution Summary:
count    233154.000000
mean         34.100946
std           9.805992
min          18.000000
25%          26.000000
50%          32.000000
75%          41.000000
max          69.000000
Name: age, dtype: float64


### Step 3: Parsing Account Tenure Strings into Total Months

 Features like `'1yrs 11mon'` are unreadable by ML algorithms. We parse them into integer months: `years * 12 + months`.

In [3]:
def parse_duration_to_months(s):
    parts = s.split('yrs')
    years = int(parts[0].strip())
    months = int(parts[1].replace('mon', '').strip())
    return years * 12 + months

df['avg_acct_age_months'] = df['AVERAGE.ACCT.AGE'].apply(parse_duration_to_months)
df['credit_hist_len_months'] = df['CREDIT.HISTORY.LENGTH'].apply(parse_duration_to_months)

print("Account Tenure Features (in Months):")
display(df[['avg_acct_age_months', 'credit_hist_len_months']].describe())

Account Tenure Features (in Months):


,avg_acct_age_months,credit_hist_len_months
count,233154.000000,233154.000000
mean,8.915764,16.252404
std,15.106416,28.581255
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,13.000000,24.000000
max,369.000000,468.000000


### 4: Domain Financial Engineering

- **`is_new_to_credit`:** As seen in EDA, over 50% of applicants have 0 bureau score. An explicit flag tells linear and tree models that 0 represents lack of credit history, not a bad score.
- **`down_payment`:** Represents applicant skin-in-the-game (`asset_cost - disbursed_amount`). Higher down payment generally correlates with lower default probability.

In [4]:
# 1. New to Credit Flag
df['is_new_to_credit'] = (df['PERFORM_CNS.SCORE'] == 0).astype(int)

# 2. Down Payment Amount (Borrower Equity)
df['down_payment'] = df['asset_cost'] - df['disbursed_amount']

# 3. Disbursal Month
df['disbursal_month'] = df['DisbursalDate'].apply(lambda x: int(x.split('-')[1]))

print(f"New-to-Credit Borrowers: {df['is_new_to_credit'].sum():,} ({df['is_new_to_credit'].mean():.1%})")
print(f"Average Down Payment:    INR {df['down_payment'].mean():,.2f}")

New-to-Credit Borrowers: 116,950 (50.2%)
Average Down Payment:    INR 21,508.07


### Step 5: Bureau Risk Tier Consolidation

Consolidates 20 raw CRISIL text categories into 6 clean, standardized risk tiers.

In [5]:
def map_bureau_risk(desc):
    if 'Very High Risk' in desc:
        return 'Very High'
    elif 'High Risk' in desc:
        return 'High'
    elif 'Medium Risk' in desc:
        return 'Medium'
    elif 'Very Low Risk' in desc:
        return 'Very Low'
    elif 'Low Risk' in desc:
        return 'Low'
    else:
        return 'No History'

df['bureau_risk_tier'] = df['PERFORM_CNS.SCORE.DESCRIPTION'].apply(map_bureau_risk)
print("Consolidated Bureau Risk Tiers vs Default Rates:")
display(df.groupby('bureau_risk_tier')['loan_default'].agg(['count', 'mean']).rename(columns={'mean': 'default_rate'}))

Consolidated Bureau Risk Tiers vs Default Rates:


,count,default_rate
bureau_risk_tier,,
High,12025,0.270104
Low,18294,0.183995
Medium,12412,0.255640
No History,129785,0.229726
Very High,9910,0.301816
Very Low,50728,0.158059


### Step 6: One-Hot Encoding & Final Feature Matrix Assembly

Drop original text strings and apply One-Hot Encoding (`drop_first=True` to prevent multicollinearity) for categorical features.

In [6]:
# Drop original unparsed string columns
cols_to_drop = [
    'Date.of.Birth',
    'DisbursalDate',
    'AVERAGE.ACCT.AGE',
    'CREDIT.HISTORY.LENGTH',
    'PERFORM_CNS.SCORE.DESCRIPTION'
]
df_features = df.drop(columns=cols_to_drop)

# One-Hot Encode categorical variables
df_features = pd.get_dummies(
    df_features,
    columns=['Employment.Type', 'bureau_risk_tier'],
    drop_first=True,
    dtype=int
)

print(f"Final Feature Matrix Shape: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns")
print(f"Remaining Null Values:      {df_features.isnull().sum().sum()}")
print("\nDatatype Breakdown:")
print(df_features.dtypes.value_counts())
df_features.head(2)

Final Feature Matrix Shape: 233,154 rows x 44 columns
Remaining Null Values:      0

Datatype Breakdown:
int64      43
float64     1
Name: count, dtype: int64


,disbursed_amount,asset_cost,ltv,branch_id,manufacturer_id,State_ID,MobileNo_Avl_Flag,Aadhar_flag,PAN_flag,VoterID_flag,Driving_flag,Passport_flag,PERFORM_CNS.SCORE,PRI.NO.OF.ACCTS,PRI.ACTIVE.ACCTS,PRI.OVERDUE.ACCTS,PRI.CURRENT.BALANCE,PRI.SANCTIONED.AMOUNT,PRI.DISBURSED.AMOUNT,SEC.NO.OF.ACCTS,SEC.ACTIVE.ACCTS,SEC.OVERDUE.ACCTS,SEC.CURRENT.BALANCE,SEC.SANCTIONED.AMOUNT,SEC.DISBURSED.AMOUNT,PRIMARY.INSTAL.AMT,SEC.INSTAL.AMT,NEW.ACCTS.IN.LAST.SIX.MONTHS,DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS,NO.OF_INQUIRIES,loan_default,age,avg_acct_age_months,credit_hist_len_months,is_new_to_credit,down_payment,disbursal_month,Employment.Type_Salaried,Employment.Type_Self employed,bureau_risk_tier_Low,bureau_risk_tier_Medium,bureau_risk_tier_No History,bureau_risk_tier_Very High,bureau_risk_tier_Very Low
0,50578,58400,89.55,67,45,6,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,34,0,0,1,7822,8,1,0,0,0,1,0,0
1,47145,65550,73.23,67,45,6,1,1,0,0,0,0,598,1,1,1,27600,50200,50200,0,0,0,0,0,0,1991,0,0,1,0,1,33,23,23,0,18405,9,0,1,0,1,0,0,0


### Step 7: Export Cleaned Feature Matrix

Save the engineered dataset to `data/train_features.csv` for Phase 4 (Model Training & Comparison).

In [7]:
OUTPUT_PATH = Path("../data/train_features.csv") if Path("../data").exists() else Path("data/train_features.csv")
df_features.to_csv(OUTPUT_PATH, index=False)
print(f"Successfully saved ML-ready feature matrix to: {OUTPUT_PATH.resolve()}")

Successfully saved ML-ready feature matrix to: C:\Users\Ratul\Desktop\ml prjct\Loan\data\train_features.csv
